<a href="https://colab.research.google.com/github/camiloascencio-hash/GP-S/blob/main/Entrega2_BdM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Caso integrador Bebidas del Maipo S.A. — Entrega 2: planificación de producción e inventario
**Gestión de Producción de Bienes y Servicios · USACH · 2.º semestre 2026 · Grupo 7**

El cuaderno tiene dos partes. Las secciones 1 a 5 son el **puente**: no formulan ni resuelven el plan, solo dejan los
datos listos para la Entrega 2. La sección 6 construye el **modelo de lot-sizing capacitado**, elemento por elemento
(§7.1 del enunciado), cada uno con su prueba.

El puente hace tres cosas:

1. Lee los **traspasos de la Entrega 1** desde `resultados/` del repositorio (salida de `Entrega1_BdM_v2.ipynb`) y
   verifica que estén completos y que sean los corregidos.
2. Lee los **datos de planificación** de `datos/`, revisa su consistencia y deja en una bitácora lo que no cuadra o
   falta.
3. Junta todo en **tablas ordenadas** (por SKU, por SKU y semana, por recurso y semana, y de preparaciones) en
   `resultados_E2/`, para que el modelo las lea directamente.

En el puente no se toma ninguna decisión de modelado: los valores son los de los datos, los del enunciado o los que la
Entrega 1 dejó comprometidos en su tabla de traspaso. Las decisiones aparecen en la sección 6 y quedan marcadas.

| Sección | Contenido |
|---|---|
| 1 | Configuración y carga del repositorio |
| 2 | Traspasos de la E1 y su verificación |
| 3 | Datos de planificación y bitácora |
| 4 | Tablas para el modelo |
| 5 | Comprobaciones y exportación |
| 6 | Modelo, Parte A: elemento 1 (balance), elemento 2 (preparación y cota M), elemento 3 (capacidad de envasado y mezcla) |

## 1. Configuración y carga del repositorio

In [ ]:
import os, json, hashlib, subprocess, platform, warnings
from importlib import metadata
import numpy as np
import pandas as pd

warnings.simplefilter("default")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

CONFIG = {
    "repo": "https://github.com/camiloascencio-hash/GP-S.git",
    "carpeta_datos": "GP-S/datos",
    "carpeta_E1": "GP-S/resultados",        # traspasos de la Entrega 1 (v2), tal como están en el repositorio
    "carpeta_salida": "resultados_E2",
    "T": 13,                                # horizonte táctico (semanas), enunciado §7.1
    "semana_focal": 12,                     # semana de los pedidos en firme (14/12/2026), enunciado §7.1
    # Datos del enunciado que no vienen en ningún CSV (§2.2 a §2.4)
    "costo_hora_cip_clp": 74_000,
    "costo_cambio_formato_clp": 38_000,
    "horas_cambio_formato": {"L1": 0.4, "L2": 0.6},
    "tol": 1e-6,
}
T, FOCAL, TOL = CONFIG["T"], CONFIG["semana_focal"], CONFIG["tol"]
SAL, E1, DAT = CONFIG["carpeta_salida"], CONFIG["carpeta_E1"], CONFIG["carpeta_datos"]

if not os.path.isdir(DAT):
    subprocess.run(["git", "clone", "-q", CONFIG["repo"]], check=True)
assert os.path.isdir(DAT) and os.path.isdir(E1), "No se encontró el repositorio"
os.makedirs(SAL, exist_ok=True)

versiones = {"python": platform.python_version(), **{p: metadata.version(p) for p in ["numpy", "pandas"]}}
print("Versiones:", versiones)

chequeos = []
def chequear(descripcion, condicion, detalle=""):
    assert bool(condicion), "FALLÓ: " + descripcion + (f" — {detalle}" if detalle else "")
    chequeos.append({"chequeo": descripcion, "resultado": "OK", "detalle": detalle})

def sha256(ruta):
    with open(ruta, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()

Versiones: {'python': '3.13.18', 'numpy': '2.5.3', 'pandas': '3.0.5'}


## 2. Traspasos de la Entrega 1 y su verificación

La Entrega 1 dejó comprometidos cuatro parámetros (§6.3): merma $\rho_\ell$ por línea (C1), jarabe extra por caja por
sobrellenado (C2), pronóstico $\hat d_{it}$ (C3) y desviación del error $\hat\sigma_{i,h}$ (C4), más el sesgo
$\hat b_{i,h}$ (C4b). Se usan **los valores de la tabla de traspaso consolidada**, que es la que la E1 entregó como
entrada de la E2. Se verifica que estén completos y que el σ̂ sea el corregido: si el repositorio tuviera el de la v1
(RMSE suavizado), el cuaderno se detiene. Se guarda el hash SHA-256 de cada archivo para rastrear cualquier cifra de
la E2 hasta el archivo exacto de la E1.

In [ ]:
archivos_E1 = ["traspaso_C1_merma.csv", "traspaso_C2_sobrellenado.csv", "traspaso_C3_pronostico.csv",
               "traspaso_C4_sigma.csv", "traspaso_C4b_sesgo.csv", "traspaso_consolidado.csv",
               "referencia_v1_rmse_suavizado.csv", "comprobaciones.csv", "config_ejecucion.json"]
for a in archivos_E1:
    chequear(f"Existe {a} en la carpeta de la E1", os.path.exists(os.path.join(E1, a)))

c3 = pd.read_csv(os.path.join(E1, "traspaso_C3_pronostico.csv"), index_col="t")
c4 = pd.read_csv(os.path.join(E1, "traspaso_C4_sigma.csv"), index_col="h")
c4b = pd.read_csv(os.path.join(E1, "traspaso_C4b_sesgo.csv"), index_col="h")
c4_v1 = pd.read_csv(os.path.join(E1, "referencia_v1_rmse_suavizado.csv"), index_col="h")
consolidado = pd.read_csv(os.path.join(E1, "traspaso_consolidado.csv"))
comp_E1 = pd.read_csv(os.path.join(E1, "comprobaciones.csv"))
with open(os.path.join(E1, "config_ejecucion.json"), encoding="utf-8") as f:
    config_E1 = json.load(f)

maestro = pd.read_csv(os.path.join(DAT, "maestro_sku.csv")).set_index("sku")
skus = list(maestro.index)

for nombre, tab in [("C3 pronóstico", c3), ("C4 σ̂", c4), ("C4b sesgo", c4b)]:
    chequear(f"{nombre}: 13 semanas × 9 SKU, sin vacíos", tab.shape == (T, len(skus)) and set(tab.columns) == set(skus)
             and list(tab.index) == list(range(1, T + 1)) and not tab.isna().any().any())
chequear("C3 pronóstico no negativo", (c3 >= 0).all().all())
chequear("C4 σ̂ finito y positivo", np.isfinite(c4.values).all() and (c4.values > 0).all())
dif_v1 = (c4[skus] - c4_v1[skus]).abs().max().max()
chequear("C4 es el σ̂ corregido (v2), no el RMSE suavizado de la v1", dif_v1 > 1, f"diferencia máxima con la v1: {dif_v1:.0f} cajas")
for p, tab in [("C3 pronóstico", c3), ("C4 sigma del error", c4), ("C4b sesgo del error", c4b)]:
    d_ = consolidado[consolidado["parametro"] == p].pivot(index="t_o_h", columns="sku", values="valor")
    chequear(f"Consolidado de la E1 = tabla {p.split()[0]} (117 registros)", len(consolidado[consolidado["parametro"] == p]) == 117
             and np.allclose(d_.loc[tab.index, skus].values, tab[skus].values, atol=TOL))
chequear("Las comprobaciones de la E1 están todas en OK", (comp_E1["resultado"] == "OK").all(), f"{len(comp_E1)} comprobaciones")

# Merma y sobrellenado: el valor que la E1 dejó en la tabla de traspaso consolidada
rho = consolidado[consolidado["parametro"] == "C1 rho (merma efectiva)"].set_index("linea")["valor"]
L_extra = consolidado[consolidado["parametro"] == "C2 jarabe extra por caja"].set_index("formato_L")["valor"]
L_extra.index = np.round(L_extra.index.astype(float), 1)
chequear("Merma ρ para L1 y L2, en (0, 1)", set(rho.index) == {"L1", "L2"} and rho.between(0, 1, inclusive="neither").all())
chequear("Jarabe extra por caja para los tres formatos", set(L_extra.index) == {0.5, 1.5, 3.0} and (L_extra >= 0).all())

hashes_E1 = {a: sha256(os.path.join(E1, a)) for a in archivos_E1}
print("Merma ρ por línea:", rho.round(4).to_dict())
print("Jarabe extra por caja (L), por formato:", L_extra.round(4).to_dict())
print("σ̂ medio por SKU (v2) frente a la v1, en cajas:")
display(pd.DataFrame({"σ̂ v2": c4[skus].mean(), "v1 (RMSE suavizado)": c4_v1[skus].mean()}).round(0).T)
print("Versiones de la ejecución de la E1:", config_E1.get("versiones"))

Merma ρ por línea: {'L1': 0.0192, 'L2': 0.0311}
Jarabe extra por caja (L), por formato: {0.5: 0.0686, 1.5: 0.0279, 3.0: 0.0323}
σ̂ medio por SKU (v2) frente a la v1, en cajas:


,NEC-D05,NEC-D15,NEC-D30,CIT-L05,CIT-L15,CIT-L30,FUN-B05,FUN-B15,FUN-B30
σ̂ v2,423.0,725.0,163.0,441.0,437.0,102.0,475.0,557.0,360.0
v1 (RMSE suavizado),474.0,727.0,180.0,460.0,685.0,119.0,608.0,687.0,354.0


Versiones de la ejecución de la E1: {'python': '3.13.18', 'numpy': '2.4.6', 'pandas': '2.2.3', 'scipy': '1.16.3', 'statsmodels': '0.15.0', 'matplotlib': '3.10.0'}


## 3. Datos de planificación y bitácora

Se usan cinco archivos de `datos/`. Los de calidad y la demanda histórica ya quedaron resumidos en los traspasos de
la E1. Cada revisión queda en la bitácora con archivo, chequeo, resultado y observación. Lo que falta o es ambiguo
**se deja anotado como pendiente**: se resuelve al formular el modelo.

In [ ]:
cap = pd.read_csv(os.path.join(DAT, "capacidad_horizonte.csv")).set_index("t")
inv = pd.read_csv(os.path.join(DAT, "inventario_inicial.csv")).set_index("sku")
cip = pd.read_csv(os.path.join(DAT, "setups_cip.csv"))
firmes_raw = pd.read_csv(os.path.join(DAT, "compromisos_semana_focal.csv"))
for nombre, d_ in [("maestro_sku", maestro), ("capacidad_horizonte", cap), ("inventario_inicial", inv),
                   ("setups_cip", cip), ("compromisos_semana_focal", firmes_raw)]:
    print(f"{nombre:26s} {d_.shape[0]:3d} filas x {d_.shape[1]:2d} columnas")

bitacora = []
def registrar(archivo, chequeo, resultado, observacion):
    bitacora.append({"archivo": archivo, "chequeo": chequeo, "resultado": resultado, "observacion": observacion})

# ---------- capacidad_horizonte
chequear("capacidad_horizonte: 13 semanas = horizonte de la E1", list(cap.index) == list(range(1, T + 1)))
registrar("capacidad_horizonte.csv", "Horizonte", f"{len(cap)} semanas, {cap['fecha_lunes'].iloc[0]} a {cap['fecha_lunes'].iloc[-1]}",
          "Coincide con el pronóstico de la E1 (t = 1 a 13)")
nominal = {"mezcla": 55, "L2": 48}
for rec, col_reg, col_man in [("mezcla", "horas_mezcla_regular", "mantencion_programada_h_mezcla"),
                              ("L2", "horas_L2_regular", "mantencion_programada_h_L2")]:
    semanas_man = cap.index[cap[col_man] > 0].tolist()
    ya_descontada = np.allclose(cap.loc[semanas_man, col_reg] + cap.loc[semanas_man, col_man], nominal[rec])
    registrar("capacidad_horizonte.csv", f"Mantenciones de {rec}",
              f"semanas {semanas_man}: {[float(v) for v in cap.loc[semanas_man, col_man]]} h; regular + mantención = {nominal[rec]} h",
              "Las horas regulares ya vienen descontadas: no hay que restar la mantención otra vez")
    chequear(f"Mantenciones de {rec} ya descontadas de las horas regulares", ya_descontada)
t13 = cap.loc[T]
registrar("capacidad_horizonte.csv", "Semana 13 (21/12/2026)", f"regular mezcla {t13['horas_mezcla_regular']:.0f} h, L1 "
          f"{t13['horas_L1_regular']:.0f} h, L2 {t13['horas_L2_regular']:.0f} h, sin mantención registrada",
          "Capacidad menor que la normal (55, 40 y 48 h); coincide con el feriado del viernes 25/12")
registrar("capacidad_horizonte.csv", "Costo de hora extra de L1",
          "No hay columna para L1 (solo L2 y mezcla) y el enunciado tampoco lo da, aunque permite hasta 8 h extra",
          "PENDIENTE: dato faltante, se define al formular el modelo")
for col, v in [("tasa_mezcla_L_h", 2300), ("cip_mezcla_horas", 1.2), ("costo_hora_extra_L2_clp", 168_000), ("costo_hora_extra_mezcla_clp", 121_000),
               ("horas_mezcla_extra_max", 10), ("horas_L1_extra_max", 8), ("horas_L2_extra_max", 16)]:
    ok = (cap[col] == v).all()
    registrar("capacidad_horizonte.csv", f"{col} constante e igual al enunciado ({v})", str(ok), "OK" if ok else "Se usa el valor del CSV")
    chequear(f"{col} coincide con el enunciado", ok)

# ---------- maestro_sku
enunciado_maestro = {"NEC-D05": (3840, 6620), "NEC-D15": (5760, 9930), "NEC-D30": (7680, 13240), "CIT-L05": (2820, 5320),
                     "CIT-L15": (4230, 7980), "CIT-L30": (5640, 10640), "FUN-B05": (4860, 9920), "FUN-B15": (7290, 14880),
                     "FUN-B30": (9720, 19840)}
ok = all(tuple(maestro.loc[s, ["costo_variable_caja_clp", "precio_lista_caja_clp"]]) == v for s, v in enunciado_maestro.items())
registrar("maestro_sku.csv", "Costos y precios = tabla del enunciado §2.1", str(ok), "OK")
chequear("maestro_sku coincide con el enunciado", ok)
ok = (maestro["linea_envasado"] == np.where(maestro["formato_L"] == 0.5, "L1", "L2")).all()
registrar("maestro_sku.csv", "Línea según formato (0,5 L → L1; 1,5 y 3,0 L → L2)", str(ok), "OK")
chequear("Línea de envasado coherente con el formato", ok)
ok = (maestro["tasa_envasado_cajas_h"] == maestro["formato_L"].map({0.5: 180, 1.5: 210, 3.0: 130})).all()
registrar("maestro_sku.csv", "Tasas de envasado = enunciado §2.2 (180, 210 y 130 cajas/h)", str(ok), "OK")
chequear("Tasas de envasado coinciden con el enunciado", ok)
registrar("maestro_sku.csv", "Costo de inventario y penalidad de atraso",
          f"{sorted(int(v) for v in maestro['costo_inventario_caja_semana_clp'].unique())} y "
          f"{sorted(int(v) for v in maestro['penalidad_atraso_caja_semana_clp'].unique())} CLP/caja·semana",
          "Iguales para todos los SKU, como el enunciado §2.4")
registrar("enunciado §2.3", "Cambio de formato en L1 (0,4 h)", "L1 envasa un solo formato (0,5 L), y cada familia tiene un solo SKU en L1",
          "PENDIENTE: el dato existe, pero en L1 no hay dos formatos de una misma familia entre los cuales cambiar")

# ---------- setups_cip
H_cip = cip.pivot(index="familia_origen", columns="familia_destino", values="horas_limpieza_cip")
C_cip = cip.pivot(index="familia_origen", columns="familia_destino", values="costo_cip_clp")
familias = list(H_cip.index)
ok_costo = np.allclose(cip["costo_cip_clp"], cip["horas_limpieza_cip"] * CONFIG["costo_hora_cip_clp"])
asim = [(a, b) for a in familias for b in familias if a < b and H_cip.loc[a, b] != H_cip.loc[b, a]]
registrar("setups_cip.csv", "Costo = horas × 74.000 (enunciado §2.3)", str(ok_costo), "OK")
registrar("setups_cip.csv", "Diagonal nula y asimetría", f"diagonal = 0: {bool((np.diag(H_cip) == 0).all())}; pares asimétricos: {asim}",
          "Matriz completa disponible para el modelo")
chequear("Costo CIP = horas × 74.000 y diagonal nula", ok_costo and (np.diag(H_cip) == 0).all())
chequear("Matriz CIP = enunciado §2.3", np.allclose(H_cip.loc[["CIT", "FUN", "NEC"], ["CIT", "FUN", "NEC"]].values,
                                                    [[0, 1.0, 1.0], [2.4, 0, 1.4], [2.8, 2.8, 0]]))

# ---------- inventario_inicial
chequear("inventario_inicial: los 9 SKU", set(inv.index) == set(skus))
registrar("inventario_inicial.csv", "Lead time de jarabe", f"{sorted(int(v) for v in inv['lead_time_jarabe_semanas'].unique())} semana(s) en los 9 SKU",
          "PENDIENTE: el enunciado no dice si es entre mezcla y envasado o de abastecimiento; se define al formular")
registrar("inventario_inicial.csv", "Nivel de servicio objetivo",
          f"{ {s: float(v) for s, v in inv['nivel_servicio_objetivo'].items()} }", "95 % en CIT y NEC, 92 % en FUN")
registrar("inventario_inicial.csv", "Atraso inicial", "El archivo no trae atraso pendiente al inicio del horizonte", "Solo inventario inicial")

# ---------- compromisos_semana_focal
ok = (firmes_raw["linea_envasado"] == firmes_raw["sku"].map(maestro["linea_envasado"])).all() and \
     (firmes_raw["familia"] == firmes_raw["sku"].map(maestro["familia"])).all()
registrar("compromisos_semana_focal.csv", "Línea y familia de cada orden = maestro", str(ok), "OK")
chequear("Pedidos en firme coherentes con el maestro", ok)
registrar("compromisos_semana_focal.csv", "Detalle por día y hora",
          f"{len(firmes_raw)} órdenes, días {sorted(int(v) for v in firmes_raw['dia_comprometido'].unique())}",
          "La E2 usa la suma semanal por SKU (§7.1, elemento 8); día y hora son para la E3")

maestro_sku                  9 filas x 12 columnas
capacidad_horizonte         13 filas x 13 columnas
inventario_inicial           9 filas x  4 columnas
setups_cip                   9 filas x  5 columnas
compromisos_semana_focal    15 filas x 10 columnas


## 4. Tablas para el modelo

### 4.1 Por SKU

Datos del maestro y del inventario inicial, más los dos traspasos por línea y formato de la E1. Las únicas columnas
calculadas son conversiones de unidades: horas de envasado por caja ($1/\text{tasa}$), litros de jarabe por caja
(nominales + sobrellenado de C2) y horas de mezcla por caja (litros / 2.300).

In [ ]:
tasa_mezcla = float(cap["tasa_mezcla_L_h"].iloc[0])
par_sku = pd.DataFrame(index=pd.Index(skus, name="sku"))
par_sku["familia"] = maestro["familia"]
par_sku["linea"] = maestro["linea_envasado"]
par_sku["formato_L"] = maestro["formato_L"]
par_sku["botellas_por_caja"] = maestro["botellas_por_caja"]
par_sku["tasa_cajas_h"] = maestro["tasa_envasado_cajas_h"]
par_sku["h_envasado_por_caja"] = 1 / maestro["tasa_envasado_cajas_h"]
par_sku["litros_nominales_caja"] = maestro["litros_por_caja"]
par_sku["litros_sobrellenado_caja"] = maestro["formato_L"].round(1).map(L_extra)          # traspaso C2
par_sku["litros_jarabe_caja"] = par_sku["litros_nominales_caja"] + par_sku["litros_sobrellenado_caja"]
par_sku["h_mezcla_por_caja"] = par_sku["litros_jarabe_caja"] / tasa_mezcla
par_sku["rho_merma"] = par_sku["linea"].map(rho)                                          # traspaso C1
par_sku["inventario_inicial"] = inv["inventario_inicial_cajas"]
par_sku["lote_minimo_cajas"] = inv["lote_minimo_cajas"]
par_sku["nivel_servicio"] = inv["nivel_servicio_objetivo"]
par_sku["lead_time_jarabe_semanas"] = inv["lead_time_jarabe_semanas"]
par_sku["costo_variable_caja"] = maestro["costo_variable_caja_clp"]
par_sku["precio_lista_caja"] = maestro["precio_lista_caja_clp"]
par_sku["costo_inventario_caja_sem"] = maestro["costo_inventario_caja_semana_clp"]
par_sku["penalidad_atraso_caja_sem"] = maestro["penalidad_atraso_caja_semana_clp"]
display(par_sku.round(4))

,familia,linea,formato_L,botellas_por_caja,tasa_cajas_h,h_envasado_por_caja,litros_nominales_caja,litros_sobrellenado_caja,litros_jarabe_caja,h_mezcla_por_caja,rho_merma,inventario_inicial,lote_minimo_cajas,nivel_servicio,lead_time_jarabe_semanas,costo_variable_caja,precio_lista_caja,costo_inventario_caja_sem,penalidad_atraso_caja_sem
sku,,,,,,,,,,,,,,,,,,,
NEC-D05,NEC,L1,0.5,12,180,0.0056,6.0,0.0686,6.0686,0.0026,0.0192,1024,350,0.95,1,3840,6620,95,1400
NEC-D15,NEC,L2,1.5,6,210,0.0048,9.0,0.0279,9.0279,0.0039,0.0311,3162,600,0.95,1,5760,9930,95,1400
NEC-D30,NEC,L2,3.0,4,130,0.0077,12.0,0.0323,12.0323,0.0052,0.0311,339,140,0.95,1,7680,13240,95,1400
CIT-L05,CIT,L1,0.5,12,180,0.0056,6.0,0.0686,6.0686,0.0026,0.0192,1217,350,0.95,1,2820,5320,95,1400
CIT-L15,CIT,L2,1.5,6,210,0.0048,9.0,0.0279,9.0279,0.0039,0.0311,1282,430,0.95,1,4230,7980,95,1400
CIT-L30,CIT,L2,3.0,4,130,0.0077,12.0,0.0323,12.0323,0.0052,0.0311,255,70,0.95,1,5640,10640,95,1400
FUN-B05,FUN,L1,0.5,12,180,0.0056,6.0,0.0686,6.0686,0.0026,0.0192,2135,550,0.92,1,4860,9920,95,1400
FUN-B15,FUN,L2,1.5,6,210,0.0048,9.0,0.0279,9.0279,0.0039,0.0311,1261,360,0.92,1,7290,14880,95,1400
FUN-B30,FUN,L2,3.0,4,130,0.0077,12.0,0.0323,12.0323,0.0052,0.0311,155,30,0.92,1,9720,19840,95,1400


### 4.2 Por SKU y semana

Pronóstico $\hat d_{it}$ (C3), σ̂ y sesgo b̂ al horizonte $h = t$ (C4 y C4b) y pedidos en firme de la semana 12,
sumados por SKU. Todo en cajas vendibles.

In [ ]:
firmes = firmes_raw.groupby("sku")["cantidad_comprometida_cajas"].sum().reindex(skus, fill_value=0)
par_sku_sem = pd.DataFrame([{"sku": i, "t": t, "fecha_lunes": cap.loc[t, "fecha_lunes"], "demanda": c3.loc[t, i],
                             "sigma": c4.loc[t, i], "sesgo": c4b.loc[t, i],
                             "firmes": float(firmes[i]) if t == FOCAL else 0.0} for i in skus for t in range(1, T + 1)])
resumen_firmes = pd.DataFrame({"firmes semana 12": firmes, "pronóstico semana 12": c3.loc[FOCAL, skus],
                               "firmes / pronóstico %": 100 * firmes / c3.loc[FOCAL, skus]})
display(resumen_firmes.round(1))
chequear("Pedidos en firme ≤ pronóstico de la semana 12 en todos los SKU", (firmes <= c3.loc[FOCAL, skus] + TOL).all())
registrar("compromisos_semana_focal.csv", "Pedidos en firme frente al pronóstico de la semana 12",
          f"entre {resumen_firmes['firmes / pronóstico %'].min():.0f} % y {resumen_firmes['firmes / pronóstico %'].max():.0f} % del pronóstico",
          "Ningún SKU tiene más pedidos firmes que pronóstico")

,firmes semana 12,pronóstico semana 12,firmes / pronóstico %
NEC-D05,898,1341.7,66.9
NEC-D15,1320,2370.6,55.7
NEC-D30,313,548.0,57.1
CIT-L05,839,2585.3,32.5
CIT-L15,1026,2904.5,35.3
CIT-L30,157,568.8,27.6
FUN-B05,1340,2432.7,55.1
FUN-B15,860,1729.8,49.7
FUN-B30,66,226.0,29.2


### 4.3 Por recurso y semana

Horas regulares (ya sin mantención), máximo de horas extra y costo de la hora extra para la mezcla (MEZ) y las dos
líneas. El costo de la hora extra de L1 queda **vacío** porque no está en los datos (ver bitácora).

In [ ]:
filas = []
for t in cap.index:
    r = cap.loc[t]
    filas += [
        {"t": t, "fecha_lunes": r["fecha_lunes"], "recurso": "MEZ", "h_regular": r["horas_mezcla_regular"], "h_extra_max": r["horas_mezcla_extra_max"],
         "costo_hora_extra": r["costo_hora_extra_mezcla_clp"], "mantencion_h": r["mantencion_programada_h_mezcla"]},
        {"t": t, "fecha_lunes": r["fecha_lunes"], "recurso": "L1", "h_regular": r["horas_L1_regular"], "h_extra_max": r["horas_L1_extra_max"],
         "costo_hora_extra": np.nan, "mantencion_h": 0.0},
        {"t": t, "fecha_lunes": r["fecha_lunes"], "recurso": "L2", "h_regular": r["horas_L2_regular"], "h_extra_max": r["horas_L2_extra_max"],
         "costo_hora_extra": r["costo_hora_extra_L2_clp"], "mantencion_h": r["mantencion_programada_h_L2"]}]
par_cap = pd.DataFrame(filas)
display(par_cap.pivot(index="t", columns="recurso", values=["h_regular", "h_extra_max"]))

h_regular             h_extra_max            
recurso        L1    L2   MEZ          L1    L2   MEZ
t                                                    
1            40.0  48.0  55.0         8.0  16.0  10.0
2            40.0  48.0  55.0         8.0  16.0  10.0
3            40.0  48.0  55.0         8.0  16.0  10.0
4            40.0  40.0  55.0         8.0  16.0  10.0
5            40.0  48.0  55.0         8.0  16.0  10.0
6            40.0  48.0  49.0         8.0  16.0  10.0
7            40.0  48.0  55.0         8.0  16.0  10.0
8            40.0  48.0  55.0         8.0  16.0  10.0
9            40.0  40.0  55.0         8.0  16.0  10.0
10           40.0  48.0  55.0         8.0  16.0  10.0
11           40.0  48.0  55.0         8.0  16.0  10.0
12           40.0  48.0  55.0         8.0  16.0  10.0
13           36.0  40.0  47.0         8.0  16.0  10.0

### 4.4 Preparaciones

Matriz completa de CIP de envasado entre familias (horas y costo), CIP de mezcla entre familias (1,2 h) y cambio de
formato dentro de una familia (0,4 h en L1, 0,6 h en L2, 38.000 CLP). Se entregan tal como vienen; cómo representarlas
en el modelo se decide al formular.

In [ ]:
print("CIP de envasado (horas), fila = familia que sale, columna = familia que entra:")
display(H_cip)
par_prep = pd.DataFrame([
    {"concepto": "CIP de mezcla entre familias", "recurso": "MEZ", "horas": float(cap["cip_mezcla_horas"].iloc[0]),
     "costo_clp": float(cap["cip_mezcla_horas"].iloc[0]) * CONFIG["costo_hora_cip_clp"], "fuente": "capacidad_horizonte.csv y §2.3"},
    {"concepto": "Cambio de formato en la misma familia", "recurso": "L1", "horas": CONFIG["horas_cambio_formato"]["L1"],
     "costo_clp": CONFIG["costo_cambio_formato_clp"], "fuente": "enunciado §2.3"},
    {"concepto": "Cambio de formato en la misma familia", "recurso": "L2", "horas": CONFIG["horas_cambio_formato"]["L2"],
     "costo_clp": CONFIG["costo_cambio_formato_clp"], "fuente": "enunciado §2.3"},
    {"concepto": "Costo de una hora de CIP", "recurso": "todos", "horas": 1.0, "costo_clp": CONFIG["costo_hora_cip_clp"], "fuente": "enunciado §2.3 y §2.4"}])
display(par_prep)

CIP de envasado (horas), fila = familia que sale, columna = familia que entra:


familia_destino,CIT,FUN,NEC
familia_origen,,,
CIT,0.0,1.0,1.0
FUN,2.4,0.0,1.4
NEC,2.8,2.8,0.0


,concepto,recurso,horas,costo_clp,fuente
0,CIP de mezcla entre familias,MEZ,1.2,88800.0,capacidad_horizonte.csv y §2.3
1,Cambio de formato en la misma familia,L1,0.4,38000.0,enunciado §2.3
2,Cambio de formato en la misma familia,L2,0.6,38000.0,enunciado §2.3
3,Costo de una hora de CIP,todos,1.0,74000.0,enunciado §2.3 y §2.4


## 5. Comprobaciones y exportación

In [ ]:
chequear("Tabla por SKU: 9 SKU sin vacíos", len(par_sku) == len(skus) and not par_sku.isna().any().any())
chequear("Tabla SKU × semana: 117 registros sin duplicados ni vacíos",
         len(par_sku_sem) == 117 and not par_sku_sem.duplicated(["sku", "t"]).any() and not par_sku_sem.isna().any().any())
chequear("Demanda = C3 de la E1 (literal)", np.allclose(par_sku_sem.pivot(index="t", columns="sku", values="demanda")[skus].values, c3[skus].values, atol=TOL))
chequear("σ̂ = C4 de la E1 (literal)", np.allclose(par_sku_sem.pivot(index="t", columns="sku", values="sigma")[skus].values, c4[skus].values, atol=TOL))
chequear("Pedidos en firme solo en la semana 12 y suma = archivo",
         par_sku_sem.query("t != @FOCAL")["firmes"].eq(0).all() and abs(par_sku_sem["firmes"].sum() - firmes_raw["cantidad_comprometida_cajas"].sum()) < TOL)
chequear("Capacidad: 13 semanas × 3 recursos", len(par_cap) == 3 * T)

par_sku.to_csv(os.path.join(SAL, "datos_sku.csv"))
par_sku_sem.to_csv(os.path.join(SAL, "datos_sku_semana.csv"), index=False)
par_cap.to_csv(os.path.join(SAL, "datos_capacidad.csv"), index=False)
H_cip.to_csv(os.path.join(SAL, "matriz_cip_envasado_horas.csv"))
C_cip.to_csv(os.path.join(SAL, "matriz_cip_envasado_costo.csv"))
par_prep.to_csv(os.path.join(SAL, "datos_preparaciones.csv"), index=False)
firmes_raw.to_csv(os.path.join(SAL, "pedidos_firmes_semana12_detalle.csv"), index=False)
pd.DataFrame(bitacora).to_csv(os.path.join(SAL, "bitacora_datos_E2.csv"), index=False)

hashes_datos = {a: sha256(os.path.join(DAT, a)) for a in ["maestro_sku.csv", "capacidad_horizonte.csv", "inventario_inicial.csv",
                                                          "setups_cip.csv", "compromisos_semana_focal.csv"]}
with open(os.path.join(SAL, "config_puente.json"), "w", encoding="utf-8") as f:
    json.dump({"config": CONFIG, "versiones": versiones, "hash_traspasos_E1": hashes_E1, "hash_datos": hashes_datos,
               "versiones_ejecucion_E1": config_E1.get("versiones"), "tasa_mezcla_L_h": tasa_mezcla,
               "familias": familias, "skus": skus}, f, indent=2, ensure_ascii=False, default=str)

tabla_chequeos = pd.DataFrame(chequeos)
tabla_chequeos.to_csv(os.path.join(SAL, "comprobaciones_puente.csv"), index=False)
print("Bitácora de datos:")
display(pd.DataFrame(bitacora))
print("Pendientes para la formulación:")
display(pd.DataFrame(bitacora).query("observacion.str.startswith('PENDIENTE')")[["archivo", "chequeo", "observacion"]])
display(tabla_chequeos)
print(f"{len(chequeos)} comprobaciones superadas.")
print("Archivos en", SAL + "/:", sorted(os.listdir(SAL)))

Bitácora de datos:


,archivo,chequeo,resultado,observacion
0,capacidad_horizonte.csv,Horizonte,"13 semanas, 2026-09-28 a 2026-12-21",Coincide con el pronóstico de la E1 (t = 1 a 13)
1,capacidad_horizonte.csv,Mantenciones de mezcla,semanas [6]: [6.0] h; regular + mantención = 55 h,Las horas regulares ya vienen descontadas: no ...
2,capacidad_horizonte.csv,Mantenciones de L2,"semanas [4, 9]: [8.0, 8.0] h; regular + manten...",Las horas regulares ya vienen descontadas: no ...
3,capacidad_horizonte.csv,Semana 13 (21/12/2026),"regular mezcla 47 h, L1 36 h, L2 40 h, sin man...","Capacidad menor que la normal (55, 40 y 48 h);..."
4,capacidad_horizonte.csv,Costo de hora extra de L1,No hay columna para L1 (solo L2 y mezcla) y el...,"PENDIENTE: dato faltante, se define al formula..."
5,capacidad_horizonte.csv,tasa_mezcla_L_h constante e igual al enunciado...,True,OK
6,capacidad_horizonte.csv,cip_mezcla_horas constante e igual al enunciad...,True,OK
7,capacidad_horizonte.csv,costo_hora_extra_L2_clp constante e igual al e...,True,OK
8,capacidad_horizonte.csv,costo_hora_extra_mezcla_clp constante e igual ...,True,OK
9,capacidad_horizonte.csv,horas_mezcla_extra_max constante e igual al en...,True,OK


Pendientes para la formulación:


,archivo,chequeo,observacion
4,capacidad_horizonte.csv,Costo de hora extra de L1,"PENDIENTE: dato faltante, se define al formula..."
16,enunciado §2.3,"Cambio de formato en L1 (0,4 h)","PENDIENTE: el dato existe, pero en L1 no hay d..."
19,inventario_inicial.csv,Lead time de jarabe,PENDIENTE: el enunciado no dice si es entre me...


,chequeo,resultado,detalle
0,Existe traspaso_C1_merma.csv en la carpeta de ...,OK,
1,Existe traspaso_C2_sobrellenado.csv en la carp...,OK,
2,Existe traspaso_C3_pronostico.csv en la carpet...,OK,
3,Existe traspaso_C4_sigma.csv en la carpeta de ...,OK,
4,Existe traspaso_C4b_sesgo.csv en la carpeta de...,OK,
5,Existe traspaso_consolidado.csv en la carpeta ...,OK,
6,Existe referencia_v1_rmse_suavizado.csv en la ...,OK,
7,Existe comprobaciones.csv en la carpeta de la E1,OK,
8,Existe config_ejecucion.json en la carpeta de ...,OK,
9,"C3 pronóstico: 13 semanas × 9 SKU, sin vacíos",OK,


45 comprobaciones superadas.
Archivos en resultados_E2/: ['bitacora_datos_E2.csv', 'comprobaciones_puente.csv', 'config_puente.json', 'datos_capacidad.csv', 'datos_preparaciones.csv', 'datos_sku.csv', 'datos_sku_semana.csv', 'matriz_cip_envasado_costo.csv', 'matriz_cip_envasado_horas.csv', 'pedidos_firmes_semana12_detalle.csv']


## Qué queda listo

| Archivo | Contenido |
|---|---|
| `datos_sku.csv` | Por SKU: familia, línea, formato, tasa y horas de envasado por caja, litros y horas de mezcla por caja (con sobrellenado), merma ρ, inventario inicial, lote mínimo, nivel de servicio, lead time y costos |
| `datos_sku_semana.csv` | Por SKU y semana: pronóstico, σ̂, sesgo y pedidos en firme (semana 12) |
| `datos_capacidad.csv` | Por recurso y semana: horas regulares, máximo de extra, costo de la hora extra y mantención |
| `matriz_cip_envasado_horas.csv`, `matriz_cip_envasado_costo.csv`, `datos_preparaciones.csv` | CIP de envasado entre familias, CIP de mezcla y cambio de formato |
| `pedidos_firmes_semana12_detalle.csv` | Las 15 órdenes en firme con día y hora (para la E3) |
| `bitacora_datos_E2.csv` | Revisión de los datos, con los pendientes marcados |
| `config_puente.json`, `comprobaciones_puente.csv` | Hashes de los archivos de entrada, versiones y comprobaciones |

## 6. Modelo de lot-sizing capacitado — Parte A: formulación

El modelo se construye elemento por elemento, en el orden de §7.1. Cada elemento lee las tablas de
`resultados_E2/` y agrega sus variables y restricciones al mismo problema `modelo`.

### 6.1 Elemento 1: balance de inventario con atraso penalizado y rendimiento $\rho_\ell$

**Punto de partida (Unidad 4, «Backlogging o demanda atrasada»).** Con inventario $s^i_t$ y atraso $b^i_t$ al cierre de
cada semana, el balance de la clase es

$$s^i_{t-1} + x^i_t + b^i_t = d^i_t + s^i_t + b^i_{t-1} \qquad \forall i, t,$$

y el atraso entra en la función objetivo con su penalización $\pi^i_t\,b^i_t$.

**Lo que agrega BdM: el rendimiento.** La Entrega 1 dejó la merma efectiva $\rho_\ell$ por línea: de cada caja que se
envasa en la línea $\ell$, solo $1-\rho_\ell$ es vendible (§6.3: «producir $x/(1-\rho_\ell)$ para obtener $x$ cajas
vendibles»). Se define $x^i_t$ como las **cajas envasadas** (brutas), porque son las que consumen horas de línea y litros
de jarabe, incluidas las defectuosas. Al inventario solo entra la parte vendible:

$$s^i_{t-1} + (1-\rho_{\ell(i)})\,x^i_t + b^i_t = d^i_t + s^i_t + b^i_{t-1} \qquad \forall i \in I,\; t = 1,\dots,13$$

**Conjuntos y datos.**

| Símbolo | Significado | Fuente |
|---|---|---|
| $I$ | 9 SKU; $\ell(i)$ es su línea de envasado (L1 para 0,5 L, L2 para 1,5 y 3,0 L) | `datos_sku.csv` |
| $t = 1,\dots,13$ | Semanas del horizonte (t = 1 es la semana del 28/09/2026) | `datos_sku_semana.csv` |
| $d^i_t$ | Pronóstico de demanda, cajas vendibles (traspaso C3) | `datos_sku_semana.csv` |
| $\rho_\ell$ | Merma efectiva: 1,92 % en L1 y 3,11 % en L2 (traspaso C1) | `datos_sku.csv` |
| $s^i_0$ | Inventario al cierre de la semana 156 | `datos_sku.csv` |
| $b^i_0$ | Atraso al inicio: 0, porque `inventario_inicial.csv` no registra atraso pendiente | bitácora |
| $h_i$ | Costo de mantener inventario: 95 CLP por caja y semana | `datos_sku.csv` |
| $\pi_i$ | Penalidad por demanda atrasada: 1.400 CLP por caja y semana | `datos_sku.csv` |

**Variables.** $x^i_t \ge 0$ cajas envasadas; $s^i_t \ge 0$ inventario vendible al cierre; $b^i_t \ge 0$ demanda atrasada
al cierre. $s$, $b$ y $d$ están en cajas vendibles; $x$ en cajas envasadas.

**Términos de la función objetivo que aporta este elemento.**

$$\sum_{i \in I}\sum_{t=1}^{13} \left(h_i\, s^i_t + \pi_i\, b^i_t\right)$$

**Cómo se lee.** Lo que había al inicio de la semana, más lo vendible que se envasa, más lo que se deja pendiente,
tiene que alcanzar para la demanda de la semana, el inventario que pasa a la siguiente y lo atrasado que venía de
antes. Como la penalidad (1.400) es casi 15 veces el costo de inventario (95), atrasar una caja una semana cuesta lo
mismo que guardarla casi 15 semanas: el modelo solo atrasará cuando no tenga capacidad para adelantar.

**Lo que este elemento deja abierto (se resuelve en los elementos siguientes).**
- **Atraso al final del horizonte $b^i_{13}$.** El balance permite terminar con demanda pendiente; en ese caso la
  penalidad se paga una sola semana y la demanda «desaparece». Se trata con la restricción de fin de horizonte
  (elemento 7).
- **Stock de seguridad** (elemento 4) y **lead time de jarabe** (elemento 6). El balance usa $x^i_t$ en la misma semana:
  si el lead time termina afectando al envasado y no solo a la mezcla, se cambiará $x^i_t$ por $x^i_{t-\gamma}$, como en la
  extensión de lead time de la clase.

In [ ]:
# ============================================================
# ELEMENTO 1 — BALANCE DE INVENTARIO CON ATRASO Y RENDIMIENTO
# ============================================================
import os
import numpy as np
import pandas as pd

# ---- 1. Instalar PuLP --------------------------------------
# PuLP no viene instalado en Colab. Se fija la versión 2.9.0 porque trae el solver CBC
# incluido y usa la sintaxis clásica (LpVariable.dicts, LpStatus). PuLP 4 cambió esa
# sintaxis y ya no incluye CBC, así que este código no correría con esa versión.
import importlib.util, importlib.metadata, subprocess, sys
VERSION_PULP = "2.9.0"
if importlib.util.find_spec("pulp") is None or importlib.metadata.version("pulp") != VERSION_PULP:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"pulp=={VERSION_PULP}"], check=True)
import pulp
assert pulp.__version__ == VERSION_PULP, f"Se esperaba PuLP {VERSION_PULP} y hay {pulp.__version__}: reinicia el entorno y vuelve a ejecutar"
SOLVER = pulp.PULP_CBC_CMD(msg=False)          # CBC sin imprimir el log del solver
print("PuLP", pulp.__version__, "| solver:", SOLVER.name)

# ---- 2. Leer los datos del puente --------------------------
# Usa la carpeta resultados_E2 de esta sesión (si se ejecutó el puente arriba)
# o, si no existe, la que está subida en el repositorio.
CARPETA_E2 = "resultados_E2" if os.path.isdir("resultados_E2") else "GP-S/resultados_E2"
datos_sku = pd.read_csv(os.path.join(CARPETA_E2, "datos_sku.csv"), index_col="sku")
datos_sku_sem = pd.read_csv(os.path.join(CARPETA_E2, "datos_sku_semana.csv"))

# ---- 3. Conjuntos y parámetros -----------------------------
I = list(datos_sku.index)                                   # conjunto I: los 9 SKU
TT = sorted(datos_sku_sem["t"].unique())                    # semanas t = 1..13
d = datos_sku_sem.pivot(index="sku", columns="t", values="demanda").loc[I, TT]   # d[i, t]: pronóstico (cajas vendibles, C3)
rend = 1 - datos_sku["rho_merma"]                           # 1 − ρ_ℓ(i): fracción vendible de cada caja envasada (C1)
s0 = datos_sku["inventario_inicial"]                        # s[i, 0]: inventario al cierre de la semana 156
b0 = pd.Series(0.0, index=I)                                # b[i, 0]: atraso inicial = 0 (no hay atraso registrado)
h = datos_sku["costo_inventario_caja_sem"]                  # h_i = 95 CLP por caja y semana
pi = datos_sku["penalidad_atraso_caja_sem"]                 # π_i = 1.400 CLP por caja y semana
assert len(I) == 9 and TT == list(range(1, 14)), "Revisar las tablas de resultados_E2"

# ---- 4. Variables y restricción de balance -----------------
def crear_variables(problema, nombre, indices, **opciones):
    """Crea un diccionario de variables indexadas por (SKU, semana): var[i][t]."""
    return pulp.LpVariable.dicts(nombre, indices, **opciones)

def agregar_balance(problema):
    """Agrega al problema las variables x, s, b y el balance de inventario de cada SKU y semana.
    Se escribe como función para poder usarla en el modelo y en la prueba sin repetir código."""
    x = crear_variables(problema, "x", (I, TT), lowBound=0)     # x[i][t]: cajas ENVASADAS (brutas)
    s = crear_variables(problema, "s", (I, TT), lowBound=0)     # s[i][t]: inventario vendible al cierre de t
    b = crear_variables(problema, "b", (I, TT), lowBound=0)     # b[i][t]: demanda atrasada al cierre de t
    for i in I:
        for t in TT:
            # En t = 1 el inventario y el atraso "anteriores" son datos (s0, b0); después, son variables
            s_ant = float(s0[i]) if t == TT[0] else s[i][t - 1]
            b_ant = float(b0[i]) if t == TT[0] else b[i][t - 1]
            # s[t-1] + (1 − ρ)·x[t] + b[t] = d[t] + s[t] + b[t-1]
            problema += (s_ant + float(rend[i]) * x[i][t] + b[i][t] == float(d.loc[i, t]) + s[i][t] + b_ant), f"balance_{i}_{t}"
    # Términos de costo que aporta este elemento a la función objetivo
    costo_inventario = pulp.lpSum(float(h[i]) * s[i][t] for i in I for t in TT)    # Σ h·s
    costo_atraso = pulp.lpSum(float(pi[i]) * b[i][t] for i in I for t in TT)       # Σ π·b
    return x, s, b, {"inventario": costo_inventario, "atraso": costo_atraso}

# ---- 5. Crear el modelo y agregarle el elemento 1 ----------
# Los elementos siguientes (2 a 8) agregan sus restricciones a este mismo "modelo"
# y sus costos al diccionario "terminos_objetivo".
modelo = pulp.LpProblem("BdM_lot_sizing_capacitado", pulp.LpMinimize)
x, s, b, terminos_objetivo = agregar_balance(modelo)

print(f"Elemento 1: {len(I) * len(TT)} restricciones de balance, {3 * len(I) * len(TT)} variables (x, s, b)")
display(pd.DataFrame({"línea": datos_sku["linea"], "ρ": datos_sku["rho_merma"], "rendimiento 1 − ρ": rend,
                      "cajas a envasar por 100 vendibles": 100 / rend, "inventario inicial s₀": s0}).round(4))

PuLP 2.9.0 | solver: PULP_CBC_CMD
Elemento 1: 117 restricciones de balance, 351 variables (x, s, b)


,línea,ρ,rendimiento 1 − ρ,cajas a envasar por 100 vendibles,inventario inicial s₀
sku,,,,,
NEC-D05,L1,0.0192,0.9808,101.9594,1024
NEC-D15,L2,0.0311,0.9689,103.2147,3162
NEC-D30,L2,0.0311,0.9689,103.2147,339
CIT-L05,L1,0.0192,0.9808,101.9594,1217
CIT-L15,L2,0.0311,0.9689,103.2147,1282
CIT-L30,L2,0.0311,0.9689,103.2147,255
FUN-B05,L1,0.0192,0.9808,101.9594,2135
FUN-B15,L2,0.0311,0.9689,103.2147,1261
FUN-B30,L2,0.0311,0.9689,103.2147,155


**Prueba del elemento 1.** Antes de agregar capacidad, se resuelve solo con el balance para comprobar que está bien
escrito. Sin ninguna otra restricción el óptimo se conoce de antemano: no atrasar, gastar primero el inventario inicial
y después envasar cada semana justo la demanda dividida por el rendimiento. El único costo es mantener el inventario
inicial mientras dura: $\sum_{i,t} h_i \max\left(0,\; s^i_0 - \sum_{\tau \le t} d^i_\tau\right)$. Para que el atraso final no
«desaparezca» demanda en esta prueba, se fija $b^i_{13} = 0$ solo aquí (el modelo lo resuelve en el elemento 7). Se
verifica que el solver llegue a ese costo, que la producción vendible sea la demanda neta y que el balance cierre en
las 117 celdas (con la tolerancia numérica del solver).

In [ ]:
# ============================================================
# PRUEBA DEL ELEMENTO 1
# ============================================================
# Se arma un problema APARTE ("prueba") con solo el balance, para no tocar el modelo.
prueba = pulp.LpProblem("prueba_elemento_1", pulp.LpMinimize)
xp, sp, bp, terminos_p = agregar_balance(prueba)
prueba.setObjective(terminos_p["inventario"] + terminos_p["atraso"])     # min Σ h·s + π·b
for i in I:
    # Solo en la prueba: sin atraso al final, para que la demanda no "desaparezca"
    prueba += (bp[i][TT[-1]] == 0), f"prueba_sin_atraso_final_{i}"
prueba.solve(SOLVER)
assert pulp.LpStatus[prueba.status] == "Optimal", pulp.LpStatus[prueba.status]

# Solución en tablas (filas = SKU, columnas = semanas)
X = pd.DataFrame({t: {i: xp[i][t].value() for i in I} for t in TT})
S = pd.DataFrame({t: {i: sp[i][t].value() for i in I} for t in TT})
B = pd.DataFrame({t: {i: bp[i][t].value() for i in I} for t in TT})

# Verificación 1: el balance cierra en las 117 celdas (lado izquierdo − lado derecho ≈ 0)
cierre = (pd.concat([s0, S.iloc[:, :-1]], axis=1).values + X.mul(rend, axis=0).values + B.values
          - d.values - S.values - pd.concat([b0, B.iloc[:, :-1]], axis=1).values)
TOL_SOLVER = 1e-3                                           # cajas: tolerancia numérica del solver
assert np.abs(cierre).max() < TOL_SOLVER, "El balance no cierra"

# Verificación 2: lo vendible que se envasa = demanda total − inventario inicial
vendible_neto = (d.sum(axis=1) - s0).clip(lower=0)
assert np.allclose(X.mul(rend, axis=0).sum(axis=1), vendible_neto + S[TT[-1]], atol=TOL_SOLVER), "Producción vendible ≠ demanda neta"

# Verificación 3: el costo es el del óptimo conocido (mantener el inventario inicial mientras dura)
costo_esperado = sum(float(h[i]) * max(0.0, float(s0[i]) - float(d.loc[i, :t].sum())) for i in I for t in TT)
assert abs(pulp.value(prueba.objective) - costo_esperado) < 1.0, "El costo no coincide con el óptimo conocido"

# Verificación 4: sin capacidad que lo obligue, no debería haber atraso
assert B.values.max() < TOL_SOLVER, "La prueba no debería atrasar"

print(f"Prueba del balance: {pulp.LpStatus[prueba.status]} | costo del solver = {pulp.value(prueba.objective):,.0f} CLP "
      f"| costo esperado = {costo_esperado:,.0f} CLP | máx. error de cierre del balance = {np.abs(cierre).max():.1e} cajas")
display(pd.DataFrame({"demanda 13 sem.": d.sum(axis=1), "inventario inicial": s0,
                      "vendible envasado": X.mul(rend, axis=0).sum(axis=1), "cajas envasadas": X.sum(axis=1),
                      "cajas mermadas": X.sum(axis=1) * (1 - rend), "inventario final": S[TT[-1]],
                      "atraso total": B.sum(axis=1)}).round(0))

Prueba del balance: Optimal | costo del solver = 99,502 CLP | costo esperado = 99,502 CLP | máx. error de cierre del balance = 4.8e-05 cajas


,demanda 13 sem.,inventario inicial,vendible envasado,cajas envasadas,cajas mermadas,inventario final,atraso total
NEC-D05,15427.0,1024,14403.0,14685.0,282.0,0.0,0.0
NEC-D15,27257.0,3162,24095.0,24870.0,775.0,0.0,0.0
NEC-D30,6300.0,339,5961.0,6153.0,192.0,0.0,0.0
CIT-L05,30490.0,1217,29273.0,29846.0,574.0,0.0,0.0
CIT-L15,34253.0,1282,32971.0,34031.0,1060.0,0.0,0.0
CIT-L30,6708.0,255,6453.0,6661.0,207.0,0.0,0.0
FUN-B05,32583.0,2135,30448.0,31045.0,597.0,0.0,0.0
FUN-B15,23169.0,1261,21908.0,22612.0,704.0,0.0,0.0
FUN-B30,3027.0,155,2872.0,2965.0,92.0,0.0,0.0


### 6.2 Elemento 2: variables binarias de preparación y enlace $x^i_t \le M^i_t\, y^i_t$

**Punto de partida (Unidad 4).** El modelo base de lot-sizing de la clase agrega una variable binaria de preparación
$y_t \in \{0,1\}$ por período y la enlaza con la producción mediante $x_t \le M_t\, y_t$: si en la semana no se prepara
la línea para el producto ($y_t = 0$), no se puede producir ($x_t = 0$); si se prepara ($y_t = 1$), se puede producir
hasta $M_t$. En el modelo multiproducto con capacidad compartida (MPS capacitado) la misma idea va por producto:

$$x^i_t \le M^i_t\, y^i_t, \qquad y^i_t \in \{0,1\} \qquad \forall i \in I,\; t = 1,\dots,13$$

con la restricción de capacidad $\sum_i \alpha_{ik}\, x^i_t + \sum_i \beta_{ik}\, y^i_t \le L_{kt}$, donde $\alpha_{ik}$ es el
tiempo unitario de producción del producto $i$ en el recurso $k$ y $L_{kt}$ la capacidad disponible.

**En BdM.** $y^i_t = 1$ si la semana $t$ se prepara la línea de envasado $\ell(i)$ para el SKU $i$. Son 9 × 13 = 117
variables binarias. Las preparaciones por familia (CIP) y su tiempo $\beta$ se agregan en el elemento 3, junto con la
capacidad.

**La cota $M^i_t$ sale de la capacidad, no de un número grande.** La propia restricción de capacidad de la clase dice
cuánto es lo máximo que se puede producir: si la línea $\ell(i)$ dedicara la semana $t$ completa al SKU $i$, usando todas
sus horas regulares y todas las horas extra permitidas, envasaría

$$\alpha_{i\ell}\, x^i_t \le L_{\ell t} \;\;\Longrightarrow\;\; x^i_t \le \frac{L_{\ell t}}{\alpha_{i\ell}}
= \left(H^{reg}_{\ell t} + H^{ext}_{\ell t}\right)\cdot \text{tasa}_i \;=\; M^i_t \quad \text{[cajas envasadas]},$$

con $\alpha_{i\ell} = 1/\text{tasa}_i$ horas por caja. Ningún plan factible puede producir más que eso en esa semana, así
que la cota **nunca corta un plan factible**: solo apaga la producción cuando $y^i_t = 0$. Todo sale de los datos:
horas regulares y máximo de extra de `datos_capacidad.csv` (con las mantenciones ya descontadas) y tasa de
`datos_sku.csv`. Por eso $M^i_t$ cambia por semana: es menor en las semanas 4 y 9 en L2 (mantención) y en la semana 13
(feriado).

**Por qué importa que $M$ sea ajustada.** La clase lo plantea así: primero el modelo debe representar bien la decisión;
después importan la formulación, las relajaciones y el desempeño computacional. En la relajación lineal ($0 \le y \le 1$)
el solver puede pagar solo la fracción $y^i_t = x^i_t / M^i_t$ de una preparación. Con un $M$ arbitrario (por ejemplo,
$10^6$) esa fracción es casi cero, la relajación no ve el costo de preparar y la cota del MIP queda débil: el solver
tarda más en cerrar la brecha. La prueba de abajo lo cuantifica.

**Cómo se ajustará después.** Para el modelo no capacitado, la cota clásica es la demanda que queda por servir,
$M_t = \sum_{\tau \ge t} d_\tau$. Aquí todavía no se puede usar, porque lo que hay que producir también depende del
stock de seguridad (elemento 4) y del inventario de fin de horizonte (elemento 7). Cuando estén definidos, se
reemplaza $M^i_t$ por el mínimo entre la cota de capacidad y la de demanda, si esta es menor.

In [ ]:
# ============================================================
# ELEMENTO 2 — BINARIAS DE PREPARACIÓN Y ENLACE x ≤ M·y
# ============================================================

# ---- 1. Datos de capacidad por línea y semana --------------
datos_cap = pd.read_csv(os.path.join(CARPETA_E2, "datos_capacidad.csv"))
# Horas máximas de la línea en la semana = horas regulares (ya sin mantención) + máximo de horas extra
datos_cap["h_max"] = datos_cap["h_regular"] + datos_cap["h_extra_max"]
H_max = datos_cap.pivot(index="recurso", columns="t", values="h_max")          # filas: MEZ, L1, L2 ; columnas: t
linea = datos_sku["linea"]                                   # ℓ(i): línea de envasado de cada SKU
tasa = datos_sku["tasa_cajas_h"]                             # cajas envasadas por hora (α = 1 / tasa)

# ---- 2. Cota M[i, t] = (horas regulares + extra máximas de su línea) × tasa ----
M = pd.DataFrame({t: {i: float(H_max.loc[linea[i], t] * tasa[i]) for i in I} for t in TT}).loc[I, TT]
assert np.isfinite(M.values).all() and (M.values > 0).all(), "La cota M debe ser finita y positiva"

# ---- 3. Variables binarias y restricción de enlace ---------
def agregar_preparacion(problema, x, M_tabla=M, relajar=False):
    """Elemento 2: crea y[i][t] (1 si la semana t se prepara la línea para el SKU i) y agrega x[i][t] ≤ M[i,t]·y[i][t].
    M_tabla y relajar solo se usan en la prueba, para comparar con un M grande y con la relajación lineal."""
    if relajar:
        y = pulp.LpVariable.dicts("y", (I, TT), lowBound=0, upBound=1)        # relajación lineal: 0 ≤ y ≤ 1
    else:
        y = pulp.LpVariable.dicts("y", (I, TT), cat="Binary")                 # y ∈ {0, 1}
    for i in I:
        for t in TT:
            problema += (x[i][t] <= float(M_tabla.loc[i, t]) * y[i][t]), f"enlace_{i}_{t}"
    return y

# ---- 4. Agregar el elemento 2 al modelo --------------------
y = agregar_preparacion(modelo, x)

n_bin = sum(1 for v in modelo.variables() if v.cat == "Integer" and v.name.startswith("y_"))
n_enl = sum(1 for nombre in modelo.constraints if nombre.startswith("enlace_"))
print(f"Elemento 2: {n_bin} variables binarias y, {n_enl} restricciones de enlace x ≤ M·y")
assert n_bin == len(I) * len(TT) and n_enl == len(I) * len(TT)

necesidad_max = d.div(rend, axis=0).max(axis=1)              # mayor demanda semanal del SKU, en cajas envasadas
display(pd.DataFrame({"línea": linea, "tasa (cajas/h)": tasa,
                      "M mínima (cajas)": M.min(axis=1), "M máxima (cajas)": M.max(axis=1),
                      "mayor demanda semanal / (1 − ρ)": necesidad_max.round(0),
                      "M mínima / mayor demanda": (M.min(axis=1) / necesidad_max).round(1)}))
print("Horas máximas por línea y semana (regular + extra) usadas en M:")
display(H_max.loc[["L1", "L2"]])

Elemento 2: 117 variables binarias y, 117 restricciones de enlace x ≤ M·y


,línea,tasa (cajas/h),M mínima (cajas),M máxima (cajas),mayor demanda semanal / (1 − ρ),M mínima / mayor demanda
NEC-D05,L1,180,7920.0,8640.0,1368.0,5.8
NEC-D15,L2,210,11760.0,13440.0,2447.0,4.8
NEC-D30,L2,130,7280.0,8320.0,566.0,12.9
CIT-L05,L1,180,7920.0,8640.0,2854.0,2.8
CIT-L15,L2,210,11760.0,13440.0,3246.0,3.6
CIT-L30,L2,130,7280.0,8320.0,636.0,11.5
FUN-B05,L1,180,7920.0,8640.0,3424.0,2.3
FUN-B15,L2,210,11760.0,13440.0,2465.0,4.8
FUN-B30,L2,130,7280.0,8320.0,322.0,22.6


Horas máximas por línea y semana (regular + extra) usadas en M:


t,1,2,3,4,5,6,7,8,9,10,11,12,13
recurso,,,,,,,,,,,,,
L1,48.0,48.0,48.0,48.0,48.0,48.0,48.0,48.0,48.0,48.0,48.0,48.0,44.0
L2,64.0,64.0,64.0,56.0,64.0,64.0,64.0,64.0,56.0,64.0,64.0,64.0,56.0


**Prueba del elemento 2.** Las pruebas usan un problema aparte (no tocan el modelo) con el balance del elemento 1, el
enlace de este elemento y, solo para la prueba, dos cosas más: una capacidad simple de envasado (las horas de
producción de cada línea no superan sus horas máximas) y un **costo de preparación de prueba** de 100.000 CLP por
preparación. Ese costo no es un dato del modelo: sirve solo para que el solver tenga un motivo para no preparar de
más. El costo real de preparar (CIP y cambio de formato) entra en el elemento 3. Como en la prueba del elemento 1, se
fija $b^i_{13} = 0$ para que el atraso final no haga desaparecer demanda.

1. **La cota no corta planes válidos.** El plan de la prueba del elemento 1 cabe bajo $M$ en las 117 celdas, y en
   todas las semanas $M^i_t$ supera la mayor demanda semanal del SKU.
2. **El enlace funciona.** En la solución, toda semana con producción tiene $y = 1$, toda semana con $y = 0$ tiene
   producción cero y no hay preparaciones sin producción. Además, cada SKU necesita al menos
   $\lceil \text{necesidad neta} / \max_t M^i_t \rceil$ preparaciones, porque una sola no puede producir más que $M$.
3. **Sin preparación no hay producción.** Si se prohíbe preparar CIT-L30 en las 13 semanas, el problema es
   infactible: su inventario inicial (255 cajas) no alcanza para su demanda y no se permite terminar con atraso.
4. **Por qué no un $M$ arbitrario.** Se resuelve el mismo problema con $M = 10^6$. Como la capacidad de la prueba es la
   misma, el óptimo entero tiene que ser igual; lo que cambia es la **relajación lineal**: mientras más cerca quede del
   óptimo entero, más fuerte es la formulación. La celda resuelve dos MIP y tarda unos segundos.

In [ ]:
# ============================================================
# PRUEBA DEL ELEMENTO 2
# ============================================================
import time
TOL_SOLVER = 1e-3
COSTO_PREPARACION_PRUEBA = 100_000        # CLP por preparación: SOLO para la prueba (el costo real va en el elemento 3)

# ---- Prueba 1: la cota no corta el plan del elemento 1 -----
# X es la solución de la prueba del elemento 1 (balance sin capacidad); con y = 1 tiene que caber bajo M
assert (X.values <= M.values + TOL_SOLVER).all(), "M corta el plan de la prueba del elemento 1"
assert (M.min(axis=1) >= necesidad_max).all(), "En alguna semana M es menor que la mayor demanda semanal"
print(f"Prueba 1 OK: el plan del elemento 1 cabe bajo M en las {len(I) * len(TT)} celdas "
      f"(holgura mínima {(M - X).values.min():,.0f} cajas)")

# ---- Problema de prueba: balance + enlace + capacidad simple + costo de preparación de prueba ----
def problema_prueba(nombre, M_tabla=M, relajar=False, prohibir=None):
    p = pulp.LpProblem(nombre, pulp.LpMinimize)
    xq, sq, bq, costos = agregar_balance(p)                                 # elemento 1
    yq = agregar_preparacion(p, xq, M_tabla=M_tabla, relajar=relajar)      # elemento 2
    for l in ["L1", "L2"]:                                                 # capacidad simple, solo para la prueba
        for t in TT:
            p += (pulp.lpSum(xq[i][t] * (1.0 / float(tasa[i])) for i in I if linea[i] == l)
                  <= float(H_max.loc[l, t])), f"cap_prueba_{l}_{t}"
    for i in I:
        p += (bq[i][TT[-1]] == 0), f"sin_atraso_final_{i}"                 # no terminar con demanda pendiente
    if prohibir is not None:
        for t in TT:
            p += (yq[prohibir][t] == 0), f"prohibido_{prohibir}_{t}"       # no se puede preparar este SKU
    p.setObjective(costos["inventario"] + costos["atraso"]
                   + COSTO_PREPARACION_PRUEBA * pulp.lpSum(yq[i][t] for i in I for t in TT))
    inicio = time.time()
    p.solve(SOLVER)
    return p, xq, yq, time.time() - inicio

# ---- Prueba 2: el enlace funciona --------------------------
p2, x2, y2, seg2 = problema_prueba("prueba_elemento_2_enlace")
assert pulp.LpStatus[p2.status] == "Optimal", pulp.LpStatus[p2.status]
X2 = pd.DataFrame({t: {i: x2[i][t].value() for i in I} for t in TT})
Y2 = pd.DataFrame({t: {i: round(y2[i][t].value()) for i in I} for t in TT})
assert ((X2 > TOL_SOLVER) <= (Y2 == 1)).all().all(), "Hay producción sin preparación"
assert (X2.values <= M.values * Y2.values + TOL_SOLVER).all(), "No se cumple x ≤ M·y"
assert ((Y2 == 1) <= (X2 > TOL_SOLVER)).all().all(), "Hay preparaciones sin producción"
necesidad_neta = (d.sum(axis=1) - s0).clip(lower=0) / rend                 # cajas a envasar en el horizonte
minimo_teorico = np.ceil(necesidad_neta / M.max(axis=1) - 1e-9).astype(int)
assert (Y2.sum(axis=1) >= minimo_teorico).all(), "Menos preparaciones que el mínimo teórico"
print(f"Prueba 2 OK ({seg2:.0f} s): {int(Y2.values.sum())} preparaciones; producción solo donde y = 1 y ninguna preparación vacía")
display(pd.DataFrame({"necesidad neta (cajas envasadas)": necesidad_neta.round(0), "M máxima": M.max(axis=1),
                      "preparaciones mínimas teóricas": minimo_teorico, "preparaciones del solver": Y2.sum(axis=1),
                      "semanas con producción": (X2 > TOL_SOLVER).sum(axis=1)}))

# ---- Prueba 3: sin preparación no hay producción -----------
p3, _, _, _ = problema_prueba("prueba_elemento_2_sin_preparar", prohibir="CIT-L30")
assert pulp.LpStatus[p3.status] == "Infeasible", f"Se esperaba infactible y dio {pulp.LpStatus[p3.status]}"
print(f"Prueba 3 OK: prohibir preparar CIT-L30 → {pulp.LpStatus[p3.status]} "
      f"(inventario inicial {s0['CIT-L30']} cajas, demanda de 13 semanas {d.loc['CIT-L30'].sum():,.0f})")

# ---- Prueba 4: cota de capacidad frente a un M arbitrario ---
M_grande = pd.DataFrame(1e6, index=I, columns=TT)
p4_lp_M, _, _, _ = problema_prueba("relajacion_M_capacidad", relajar=True)
p4_lp_G, _, _, _ = problema_prueba("relajacion_M_grande", M_tabla=M_grande, relajar=True)
p4_mip_G, _, _, seg4 = problema_prueba("entero_M_grande", M_tabla=M_grande)
z_mip_M, z_mip_G = pulp.value(p2.objective), pulp.value(p4_mip_G.objective)
z_lp_M, z_lp_G = pulp.value(p4_lp_M.objective), pulp.value(p4_lp_G.objective)
assert abs(z_mip_M - z_mip_G) <= 1e-4 * z_mip_M, "Con la misma capacidad, el óptimo entero no debería depender de M"
assert z_lp_M > z_lp_G, "La cota de capacidad debería dar una relajación más fuerte"
comp4 = pd.DataFrame({"óptimo entero (CLP)": [z_mip_M, z_mip_G], "relajación lineal (CLP)": [z_lp_M, z_lp_G],
                      "relajación / entero %": [100 * z_lp_M / z_mip_M, 100 * z_lp_G / z_mip_G],
                      "tiempo MIP (s)": [seg2, seg4]},
                     index=["M = capacidad (este modelo)", "M = 1.000.000 (arbitrario)"])
print("Prueba 4 OK: mismo óptimo entero, pero con M de capacidad la relajación lineal queda mucho más cerca de él")
display(comp4.round(1))

Prueba 1 OK: el plan del elemento 1 cabe bajo M en las 117 celdas (holgura mínima 5,016 cajas)


Prueba 2 OK (3 s): 91 preparaciones; producción solo donde y = 1 y ninguna preparación vacía


,necesidad neta (cajas envasadas),M máxima,preparaciones mínimas teóricas,preparaciones del solver,semanas con producción
NEC-D05,14685.0,8640.0,2,12,12
NEC-D15,24870.0,13440.0,2,12,12
NEC-D30,6153.0,8320.0,1,6,6
CIT-L05,29846.0,8640.0,4,13,13
CIT-L15,34031.0,13440.0,3,13,13
CIT-L30,6661.0,8320.0,1,6,6
FUN-B05,31045.0,8640.0,4,12,12
FUN-B15,22612.0,13440.0,2,13,13
FUN-B30,2965.0,8320.0,1,4,4


Prueba 3 OK: prohibir preparar CIT-L30 → Infeasible (inventario inicial 255 cajas, demanda de 13 semanas 6,708)


Prueba 4 OK: mismo óptimo entero, pero con M de capacidad la relajación lineal queda mucho más cerca de él


,óptimo entero (CLP),relajación lineal (CLP),relajación / entero %,tiempo MIP (s)
M = capacidad (este modelo),10354274.5,1805536.2,17.4,2.5
M = 1.000.000 (arbitrario),10354274.5,116789.0,1.1,2.8


### 6.3 Elemento 3: capacidad de envasado y de mezcla, con preparaciones y sobrellenado

**Punto de partida (Unidad 4, MPS capacitado).** La capacidad de cada recurso $k$ en cada período limita el tiempo de
producción más el tiempo fijo de las preparaciones:

$$\sum_i \alpha_{ik}\, x^i_t + \sum_i \beta_{ik}\, y^i_t \le L_{kt} \qquad \forall k, t,$$

con $\alpha_{ik}$ el tiempo unitario de producción, $\beta_{ik}$ el tiempo fijo de preparación y $L_{kt}$ la capacidad
disponible. En BdM hay tres recursos: las líneas de envasado L1 y L2 y la mezcladora (MEZ).

**Lo que agrega BdM: las preparaciones son por familia.** La limpieza CIP se hace al cambiar de **familia**, no de SKU.
Por eso, además de $y^i_t$ (elemento 2), se agregan dos variables binarias de familia activa:

- $w^f_{\ell t} = 1$ si la familia $f$ se envasa en la línea $\ell$ en la semana $t$;
- $v^f_t = 1$ si la familia $f$ se mezcla en la semana $t$.

Se activan con $w^f_{\ell t} \ge y^i_t$ y $v^f_t \ge y^i_t$ para todo SKU $i$ de la familia $f$ (y de la línea $\ell$): si se
produce un solo SKU de la familia, la familia está activa.

**Capacidad de envasado de la línea $\ell$ (en horas).**

$$\underbrace{\sum_{i \in \ell} \frac{x^i_t}{\text{tasa}_i}}_{\text{producción}}
+ \underbrace{c^{CIP}_{env} \sum_f w^f_{\ell t}}_{\text{CIP por familia activa}}
+ \underbrace{c^{fmt}_\ell \Big(\sum_{i \in \ell} y^i_t - \sum_f w^f_{\ell t}\Big)}_{\text{cambios de formato}}
\;\le\; H^{reg}_{\ell t} \qquad \forall \ell \in \{L1, L2\},\; t$$

- **CIP por familia activa.** Cada familia que entra a la línea en la semana paga una limpieza. El plan semanal decide
  cuánto producir, pero no en qué orden, así que no sabe de qué familia viene cada cambio y no puede leer la casilla
  exacta de la matriz asimétrica. Se usa un tiempo representativo $c^{CIP}_{env}$: el **promedio de la matriz fuera de la
  diagonal, 1,9 h** (parámetro `REGLA_CIP_ENVASADO`, **decisión pendiente del grupo**; el peor caso, 2,8 h, queda como
  sensibilidad). Es justo lo que la Entrega 3 pide contrastar: «suponer un tiempo de preparación promedio y enfrentar
  una matriz asimétrica».
- **Cambios de formato.** Dentro de una familia, pasar de un formato a otro no requiere CIP, pero toma 0,4 h en L1 y
  0,6 h en L2. Si en la semana hay $n$ SKU activos de una familia en la línea, hay $n - 1$ cambios de formato, y
  $\sum_i y^i_t - \sum_f w^f_{\ell t}$ cuenta exactamente eso, de forma lineal. En L1 cada familia tiene un solo SKU,
  así que ese término vale siempre cero: el dato de 0,4 h queda en el modelo, pero nunca se activa.
- $H^{reg}_{\ell t}$ son las horas regulares, ya sin mantenciones. Las **horas extra** se suman al lado derecho en el
  elemento 5.

**Capacidad de mezcla (en litros).** Cada caja envasada consume sus litros nominales más el **sobrellenado heredado de la
Entrega 1** (traspaso C2), porque la llenadora pone más jarabe del nominal. Se calcula sobre las cajas **envasadas**
$x$, porque las botellas defectuosas también llevan jarabe. Cada familia mezclada en la semana exige un CIP de 1,2 h,
en el que la mezcladora deja de producir $1{,}2 \times 2.300 = 2.760$ L:

$$\sum_{i} \underbrace{\left(L^{nom}_i + L^{sobre}_i\right)}_{\text{litros por caja}} x^i_t
\;+\; 2.300 \cdot 1{,}2 \sum_f v^f_t \;\le\; 2.300 \cdot H^{reg}_{MEZ,t} \qquad \forall t$$

**Supuesto de las semanas como bloques.** Se cobra un CIP por cada familia activa en cada semana, como pide el
enunciado («CIP por familia activa»). Es conservador: si la última familia de una semana es la primera de la
siguiente, en la realidad ese CIP no ocurre. La E3 permite medir cuánto pesa.

**Por ahora, la mezcla y el envasado de un SKU ocurren en la misma semana.** El lead time de jarabe de 1 semana se
resuelve en el elemento 6; si corresponde, ahí se desplaza la mezcla a la semana anterior.

**Términos de la función objetivo que aporta este elemento** (costo de preparación):

$$\sum_t \Big[\,74.000 \Big(c^{CIP}_{env} \sum_{\ell, f} w^f_{\ell t} + 1{,}2 \sum_f v^f_t\Big)
+ 38.000 \sum_\ell \Big(\sum_{i \in \ell} y^i_t - \sum_f w^f_{\ell t}\Big)\Big]$$

In [ ]:
# ============================================================
# ELEMENTO 3 — CAPACIDAD DE ENVASADO Y DE MEZCLA
# ============================================================

import json, time

# ---- 1. Parámetros de preparación --------------------------
REGLA_CIP_ENVASADO = "promedio"      # DECISIÓN PENDIENTE DEL GRUPO: "promedio" (1,9 h) | "maximo" (2,8 h) | "minimo" (1,0 h)

matriz_cip = pd.read_csv(os.path.join(CARPETA_E2, "matriz_cip_envasado_horas.csv"), index_col=0)
datos_prep = pd.read_csv(os.path.join(CARPETA_E2, "datos_preparaciones.csv"))
fuera_diagonal = matriz_cip.values[~np.eye(len(matriz_cip), dtype=bool)]           # los 6 cambios entre familias distintas
CIP_ENV_H = {"promedio": fuera_diagonal.mean(), "maximo": fuera_diagonal.max(), "minimo": fuera_diagonal.min()}[REGLA_CIP_ENVASADO]
CIP_MEZ_H = float(datos_prep.query("recurso == 'MEZ'")["horas"].iloc[0])          # 1,2 h por familia mezclada
FMT_H = {l: float(datos_prep.loc[datos_prep["recurso"] == l, "horas"].iloc[0]) for l in ["L1", "L2"]}   # 0,4 h y 0,6 h
COSTO_HORA_CIP = float(datos_prep.query("recurso == 'todos'")["costo_clp"].iloc[0])           # 74.000 CLP por hora
COSTO_FMT = float(datos_prep.query("recurso == 'L2'")["costo_clp"].iloc[0])                    # 38.000 CLP por cambio
with open(os.path.join(CARPETA_E2, "config_puente.json"), encoding="utf-8") as f:
    TASA_MEZCLA = float(json.load(f)["tasa_mezcla_L_h"])                                     # 2.300 L/h

familia = datos_sku["familia"]                               # f(i)
F = sorted(familia.unique())                                 # CIT, FUN, NEC
LINEAS = ["L1", "L2"]
litros_caja = datos_sku["litros_jarabe_caja"]                # litros nominales + sobrellenado (traspaso C2), por caja envasada
H_reg = datos_cap.pivot(index="recurso", columns="t", values="h_regular")              # horas regulares (sin mantención)
skus_de = {(l, f): [i for i in I if linea[i] == l and familia[i] == f] for l in LINEAS for f in F}

print(f"CIP de envasado: {REGLA_CIP_ENVASADO} de la matriz = {CIP_ENV_H:.2f} h por familia activa "
      f"(fuera de la diagonal: {sorted(float(v) for v in fuera_diagonal)})")
print(f"CIP de mezcla: {CIP_MEZ_H} h por familia mezclada | cambio de formato: L1 {FMT_H['L1']} h, L2 {FMT_H['L2']} h "
      f"| costo CIP {COSTO_HORA_CIP:,.0f} CLP/h, cambio de formato {COSTO_FMT:,.0f} CLP")

# ---- 2. Variables de familia activa y restricciones de capacidad ----
def agregar_capacidad(problema, x, y, H=H_reg, cip_env_h=CIP_ENV_H):
    """Elemento 3: crea w[l][f][t] (familia activa en la línea) y v[f][t] (familia mezclada) y agrega
    la capacidad de envasado de L1 y L2 (horas) y de mezcla (litros). H y cip_env_h solo cambian en las pruebas."""
    w = pulp.LpVariable.dicts("w", (LINEAS, F, TT), cat="Binary")
    v = pulp.LpVariable.dicts("v", (F, TT), cat="Binary")
    horas_env, litros_mez, cambios_fmt = {}, {}, {}
    for t in TT:
        # Familia activa: si se produce un SKU de la familia, la familia está activa en su línea y en la mezcla
        for i in I:
            problema += (w[linea[i]][familia[i]][t] >= y[i][t]), f"activa_env_{i}_{t}"
            problema += (v[familia[i]][t] >= y[i][t]), f"activa_mez_{i}_{t}"
        for l in LINEAS:
            for f in F:
                if not skus_de[(l, f)]:                                  # la familia no tiene SKU en esa línea
                    problema += (w[l][f][t] == 0), f"sin_sku_{l}_{f}_{t}"
        # Envasado (horas): producción + CIP por familia activa + cambios de formato ≤ horas regulares
        for l in LINEAS:
            produccion_h = pulp.lpSum(x[i][t] * (1.0 / float(tasa[i])) for i in I if linea[i] == l)
            cambios_fmt[(l, t)] = pulp.lpSum(y[i][t] for i in I if linea[i] == l) - pulp.lpSum(w[l][f][t] for f in F)
            horas_env[(l, t)] = produccion_h + float(cip_env_h) * pulp.lpSum(w[l][f][t] for f in F) + FMT_H[l] * cambios_fmt[(l, t)]
            problema += (horas_env[(l, t)] <= float(H.loc[l, t])), f"cap_env_{l}_{t}"
        # Mezcla (litros): jarabe de las cajas envasadas (con sobrellenado) + litros que se pierden en CIP ≤ litros disponibles
        litros_mez[t] = (pulp.lpSum(float(litros_caja[i]) * x[i][t] for i in I)
                         + TASA_MEZCLA * CIP_MEZ_H * pulp.lpSum(v[f][t] for f in F))
        problema += (litros_mez[t] <= TASA_MEZCLA * float(H.loc["MEZ", t])), f"cap_mez_{t}"
    # Costo de preparación: horas de CIP (envasado y mezcla) × 74.000 + cambios de formato × 38.000
    costo_prep = (COSTO_HORA_CIP * (float(cip_env_h) * pulp.lpSum(w[l][f][t] for l in LINEAS for f in F for t in TT)
                                    + CIP_MEZ_H * pulp.lpSum(v[f][t] for f in F for t in TT))
                  + COSTO_FMT * pulp.lpSum(cambios_fmt[(l, t)] for l in LINEAS for t in TT))
    return w, v, horas_env, litros_mez, costo_prep

# ---- 3. Agregar el elemento 3 al modelo --------------------
w, v, horas_env, litros_mez, terminos_objetivo["preparacion"] = agregar_capacidad(modelo, x, y)

conteo = {pref: sum(1 for n in modelo.constraints if n.startswith(pref)) for pref in ["cap_env_", "cap_mez_", "activa_env_", "activa_mez_"]}
print(f"Elemento 3: {sum(1 for v_ in modelo.variables() if v_.name.startswith(('w_', 'v_')))} binarias de familia activa (w, v); "
      f"{conteo['cap_env_']} restricciones de capacidad de envasado, {conteo['cap_mez_']} de mezcla, "
      f"{conteo['activa_env_'] + conteo['activa_mez_']} de activación")
assert conteo["cap_env_"] == 2 * len(TT) and conteo["cap_mez_"] == len(TT)
display(pd.DataFrame({"familia": familia, "línea": linea, "h envasado por caja": (1 / tasa).round(5),
                      "litros nominales": datos_sku["litros_nominales_caja"], "litros de sobrellenado (C2)": datos_sku["litros_sobrellenado_caja"],
                      "litros por caja envasada": litros_caja.round(4), "h de mezcla por caja": (litros_caja / TASA_MEZCLA).round(5)}))

CIP de envasado: promedio de la matriz = 1.90 h por familia activa (fuera de la diagonal: [1.0, 1.0, 1.4, 2.4, 2.8, 2.8])
CIP de mezcla: 1.2 h por familia mezclada | cambio de formato: L1 0.4 h, L2 0.6 h | costo CIP 74,000 CLP/h, cambio de formato 38,000 CLP
Elemento 3: 117 binarias de familia activa (w, v); 26 restricciones de capacidad de envasado, 13 de mezcla, 234 de activación


,familia,línea,h envasado por caja,litros nominales,litros de sobrellenado (C2),litros por caja envasada,h de mezcla por caja
sku,,,,,,,
NEC-D05,NEC,L1,0.00556,6.0,0.068637,6.0686,0.00264
NEC-D15,NEC,L2,0.00476,9.0,0.027873,9.0279,0.00393
NEC-D30,NEC,L2,0.00769,12.0,0.032290,12.0323,0.00523
CIT-L05,CIT,L1,0.00556,6.0,0.068637,6.0686,0.00264
CIT-L15,CIT,L2,0.00476,9.0,0.027873,9.0279,0.00393
CIT-L30,CIT,L2,0.00769,12.0,0.032290,12.0323,0.00523
FUN-B05,FUN,L1,0.00556,6.0,0.068637,6.0686,0.00264
FUN-B15,FUN,L2,0.00476,9.0,0.027873,9.0279,0.00393
FUN-B30,FUN,L2,0.00769,12.0,0.032290,12.0323,0.00523


**Prueba del elemento 3.** Igual que antes, en un problema aparte. Se arma con los elementos 1, 2 y 3 y se minimiza
inventario + atraso + preparación, con $b^i_{13} = 0$. Como las horas extra recién entran en el elemento 5, la prueba
se resuelve **dos veces**: primero solo con horas regulares, para ver si alcanzan (con un tope de 60 s, porque sin
horas extra el problema queda muy apretado y es difícil de cerrar), y después con el máximo de horas (regulares +
extra), sin cobrar la extra, solo para revisar las cuentas. Se comprueba:

1. **Las familias activas están bien:** $w$ y $v$ valen 1 exactamente cuando la familia tiene algún SKU producido.
2. **Las horas cuadran:** las horas de envasado y los litros de mezcla se recalculan a mano desde la solución (producción
   + CIP + cambios de formato) y coinciden con lo que ve el solver, sin pasar la capacidad.
3. **El costo de preparación cuadra** con el conteo de CIP y cambios de formato de la solución.
4. **El tiempo de CIP importa:** con el peor caso de la matriz (2,8 h) el plan no puede ser más barato que con el
   promedio (1,9 h).

In [ ]:
# ============================================================
# PRUEBA DEL ELEMENTO 3
# ============================================================
H_tope = H_reg + datos_cap.pivot(index="recurso", columns="t", values="h_extra_max")    # horas regulares + extra máximas

def problema_prueba_e3(nombre, H, cip_env_h=CIP_ENV_H, limite_s=60):
    p = pulp.LpProblem(nombre, pulp.LpMinimize)
    xq, sq, bq, costos = agregar_balance(p)                                          # elemento 1
    yq = agregar_preparacion(p, xq)                                                  # elemento 2
    wq, vq, hq, lq, costo_prep = agregar_capacidad(p, xq, yq, H=H, cip_env_h=cip_env_h)   # elemento 3
    for i in I:
        p += (bq[i][TT[-1]] == 0), f"sin_atraso_final_{i}"                          # solo en la prueba
    p.setObjective(costos["inventario"] + costos["atraso"] + costo_prep)
    inicio = time.time()
    p.solve(pulp.PULP_CBC_CMD(msg=False, timeLimit=limite_s))
    return p, dict(x=xq, s=sq, b=bq, y=yq, w=wq, v=vq, horas=hq, litros=lq, costo_prep=costo_prep), time.time() - inicio

def tabla(var, filas, cols):
    return pd.DataFrame({c: {r: var[r][c].value() for r in filas} for c in cols})

def estado(p):
    """Estado de la solución: 'óptimo probado', 'factible (se cortó por tiempo)' o el estado del solver."""
    return {pulp.LpSolutionOptimal: "óptimo probado", pulp.LpSolutionIntegerFeasible: "factible, se cortó por tiempo sin probar el óptimo"}.get(
        p.sol_status, pulp.LpStatus[p.status])

# ---- Primera resolución: solo horas regulares (tope de 60 s) ----
p_reg, sol_reg, seg_reg = problema_prueba_e3("prueba_e3_regulares", H_reg)
if p_reg.sol_status in (pulp.LpSolutionOptimal, pulp.LpSolutionIntegerFeasible):
    atraso_reg = sum(sol_reg["b"][i][t].value() for i in I for t in TT)
    print(f"Solo horas regulares: {estado(p_reg)} ({seg_reg:.0f} s) | costo {pulp.value(p_reg.objective):,.0f} CLP "
          f"| atraso acumulado {atraso_reg:,.0f} caja·semana → sin horas extra se cubre la demanda "
          + ("atrasando pedidos" if atraso_reg > 1 else "adelantando producción a inventario, sin atrasar"))
elif pulp.LpStatus[p_reg.status] == "Infeasible":
    print("Solo horas regulares: infactible → sin horas extra no se cubre la demanda sin terminar con atraso")
else:
    print(f"Solo horas regulares: {estado(p_reg)} en {seg_reg:.0f} s (el solver no alcanzó a encontrar un plan en el tope de tiempo)")

# ---- Segunda resolución: horas regulares + extra máximas (solo para revisar las cuentas) ----
p_e3, sol, seg_e3 = problema_prueba_e3("prueba_e3_tope", H_tope)
assert p_e3.sol_status == pulp.LpSolutionOptimal, estado(p_e3)
print(f"Con horas regulares + extra (sin cobrarlas): {estado(p_e3)} ({seg_e3:.0f} s) | costo {pulp.value(p_e3.objective):,.0f} CLP")
Xs, Ys = tabla(sol["x"], I, TT), tabla(sol["y"], I, TT).round()
Vs = tabla(sol["v"], F, TT).round()
Ws = {l: tabla(sol["w"][l], F, TT).round() for l in LINEAS}

# ---- Comprobación 1: familias activas ----------------------
for l in LINEAS:
    for f in F:
        activa_esperada = Ys.loc[skus_de[(l, f)]].max() if skus_de[(l, f)] else pd.Series(0.0, index=TT)
        assert (Ws[l].loc[f] == activa_esperada).all(), f"w mal activada en {l}, {f}"
for f in F:
    assert (Vs.loc[f] == Ys.loc[[i for i in I if familia[i] == f]].max()).all(), f"v mal activada en {f}"
print("Comprobación 1 OK: w y v valen 1 exactamente cuando la familia tiene algún SKU producido")

# ---- Comprobación 2: horas de envasado y litros de mezcla recalculados a mano ----
filas_uso = []
for t in TT:
    for l in LINEAS:
        prod_h = sum(Xs.loc[i, t] / tasa[i] for i in I if linea[i] == l)
        n_fam = Ws[l][t].sum()
        n_fmt = Ys.loc[[i for i in I if linea[i] == l], t].sum() - n_fam
        total = prod_h + CIP_ENV_H * n_fam + FMT_H[l] * n_fmt
        assert abs(total - pulp.value(sol["horas"][(l, t)])) < 1e-3 and total <= H_tope.loc[l, t] + 1e-3
        filas_uso.append({"t": t, "recurso": l, "producción h": prod_h, "CIP h": CIP_ENV_H * n_fam, "formato h": FMT_H[l] * n_fmt,
                          "total h": total, "regular h": H_reg.loc[l, t], "máximo h": H_tope.loc[l, t]})
    litros = sum(Xs.loc[i, t] * litros_caja[i] for i in I)
    total_mez = litros / TASA_MEZCLA + CIP_MEZ_H * Vs[t].sum()
    assert abs(total_mez * TASA_MEZCLA - pulp.value(sol["litros"][t])) < 1e-2 and total_mez <= H_tope.loc["MEZ", t] + 1e-3
    filas_uso.append({"t": t, "recurso": "MEZ", "producción h": litros / TASA_MEZCLA, "CIP h": CIP_MEZ_H * Vs[t].sum(), "formato h": 0.0,
                      "total h": total_mez, "regular h": H_reg.loc["MEZ", t], "máximo h": H_tope.loc["MEZ", t]})
uso = pd.DataFrame(filas_uso)
uso["sobre regular h"] = (uso["total h"] - uso["regular h"]).clip(lower=0)
print("Comprobación 2 OK: horas de envasado y litros de mezcla recalculados a mano = los del solver, sin pasar la capacidad")

# ---- Comprobación 3: costo de preparación ------------------
n_cip_env = sum(Ws[l].values.sum() for l in LINEAS)
n_fmt_tot = sum(Ys.loc[[i for i in I if linea[i] == l]].values.sum() - Ws[l].values.sum() for l in LINEAS)
costo_prep_mano = COSTO_HORA_CIP * (CIP_ENV_H * n_cip_env + CIP_MEZ_H * Vs.values.sum()) + COSTO_FMT * n_fmt_tot
assert abs(costo_prep_mano - pulp.value(sol["costo_prep"])) < 1.0
print(f"Comprobación 3 OK: costo de preparación {costo_prep_mano:,.0f} CLP = {int(n_cip_env)} CIP de envasado + "
      f"{int(Vs.values.sum())} CIP de mezcla + {int(n_fmt_tot)} cambios de formato")

# ---- Comprobación 4: sensibilidad del tiempo de CIP --------
p_max, _, seg_max = problema_prueba_e3("prueba_e3_cip_maximo", H_tope, cip_env_h=fuera_diagonal.max())
assert p_max.sol_status == pulp.LpSolutionOptimal, estado(p_max)
assert pulp.value(p_max.objective) >= pulp.value(p_e3.objective) - 1.0, "Con más horas de CIP el plan no puede salir más barato"
print(f"Comprobación 4 OK: CIP promedio ({CIP_ENV_H:.1f} h) → {pulp.value(p_e3.objective):,.0f} CLP | "
      f"CIP peor caso ({fuera_diagonal.max():.1f} h) → {pulp.value(p_max.objective):,.0f} CLP")

print("\nUso de capacidad por semana en la prueba (horas). 'sobre regular' es lo que en el modelo completo serán horas extra:")
display(uso.pivot(index="t", columns="recurso", values=["total h", "regular h", "sobre regular h"]).round(1))

Solo horas regulares: factible, se cortó por tiempo sin probar el óptimo (60 s) | costo 17,004,332 CLP | atraso acumulado 0 caja·semana → sin horas extra se cubre la demanda adelantando producción a inventario, sin atrasar


Con horas regulares + extra (sin cobrarlas): óptimo probado (2 s) | costo 15,292,977 CLP
Comprobación 1 OK: w y v valen 1 exactamente cuando la familia tiene algún SKU producido
Comprobación 2 OK: horas de envasado y litros de mezcla recalculados a mano = los del solver, sin pasar la capacidad
Comprobación 3 OK: costo de preparación 12,890,200 CLP = 63 CIP de envasado + 33 CIP de mezcla + 29 cambios de formato


Comprobación 4 OK: CIP promedio (1.9 h) → 15,292,977 CLP | CIP peor caso (2.8 h) → 18,868,790 CLP

Uso de capacidad por semana en la prueba (horas). 'sobre regular' es lo que en el modelo completo serán horas extra:


total h             regular h             sobre regular h           
recurso      L1    L2   MEZ        L1    L2   MEZ              L1    L2  MEZ
t                                                                           
1          14.3  25.1  21.8      40.0  48.0  55.0             0.0   0.0  0.0
2          30.5  29.7  35.4      40.0  48.0  55.0             0.0   0.0  0.0
3          43.0  61.3  63.5      40.0  48.0  55.0             3.0  13.3  8.5
4          30.8  28.4  34.5      40.0  40.0  55.0             0.0   0.0  0.0
5          38.8  50.2  52.8      40.0  48.0  55.0             0.0   2.2  0.0
6          47.9  49.4  57.4      40.0  48.0  49.0             7.9   1.4  8.4
7          43.4  58.2  62.6      40.0  48.0  55.0             3.4  10.2  7.6
8          28.2  46.0  46.2      40.0  48.0  55.0             0.0   0.0  0.0
9          45.3  43.4  52.6      40.0  40.0  55.0             5.3   3.4  0.0
10         33.3  54.1  55.0      40.0  48.0  55.0             0.0   6.1  0.0
11         39.1  41.1  47.6      40.0  48.0  55.0             0.0   0.0  0.0
12         48.0  50.2  58.1      40.0  48.0  55.0             8.0   2.2  3.1
13         34.4  52.7  53.9      36.0  40.0  47.0             0.0  12.7  6.9